In [ ]:
import os
# 强制 Hugging Face 使用 Rust 引擎进行极速下载/上传
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import sys
import os
from google.colab import userdata

# 1. 自动从 Colab Secrets 获取你存入的 Token
GIT_TOKEN = userdata.get('GITHUB_TOKEN')

# 2. 根据你的截图填入对应的用户名和仓库名
GIT_USER = "Sheng-Pan"
GIT_REPO = "Decentralized-federated-learning2"

# 3. 构建带认证信息的 URL
repo_url = f"https://{GIT_TOKEN}@github.com/{GIT_USER}/{GIT_REPO}.git"

# 4. 克隆仓库 (如果文件夹已存在则跳过，防止报错)
if not os.path.exists(GIT_REPO):
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists.")

# 5. 将仓库路径添加到系统路径，以便 Python 找到 functions2.py
repo_path = os.path.join("/content", GIT_REPO)
if repo_path not in sys.path:
    sys.path.append(repo_path)


Cloning into 'Decentralized-federated-learning2'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 61 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 144.03 KiB | 2.48 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [ ]:
# 4. 强制更新仓库 (如果存在则拉取最新，不存在则克隆)
import os
if not os.path.exists(GIT_REPO):
    print("Cloning new repository...")
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists. Pulling latest changes...")
    # 切换进目录更新，然后再切换出来
    %cd {GIT_REPO}
    !git pull
    %cd ..

# 5. 关键的一步：强制重新加载已经导入的模块
import importlib
import data_loader # 确保这里是 defense
importlib.reload(data_loader)
import defense # 确保这里是 defense
importlib.reload(defense)
import trainer # 确保这里是 defense
importlib.reload(trainer)
import MAB_fun # 确保这里是 defense
importlib.reload(MAB_fun)
import data_loader # 确保这里是 defense
importlib.reload(data_loader)

import main_transformer # 确保这里是 defense
importlib.reload(main_transformer)




Decentralized-federated-learning2 already exists. Pulling latest changes...
/content/Decentralized-federated-learning2
Already up to date.
/content


<module 'main_transformer' from '/content/Decentralized-federated-learning2/main_transformer.py'>

In [ ]:
from main_transformer import run_simulation_transformer
run_simulation_transformer

<function main_transformer.run_simulation_transformer(seed, NUM_CLIENTS, defense_nodes, malicious_clients, G, neighbors, client_datasets, test_data, atk_type='neurotoxin', mechanism='FedAvg', bf=1.0, intensity=0.1, debug_mode=False, GLOBAL_ROUNDS=15, scale_factor=0.2, debug=True, epochs=1)>

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import time

import torch
import gc

import glob

import copy # Add this line to import the copy module
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from defense import get_high_value_defense_nodes,count_minimum_required_defenders
from main_transformer import run_simulation_transformer
# from data_loader import set_seed # Comment out or remove this line to use the custom set_seed below
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

from data_loader import set_seed

from huggingface_hub import HfApi, snapshot_download
from google.colab import userdata

# ==========================================
# 1. Hugging Face 配置 (替代 Drive)
# ==========================================
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "JONESMITH007/DFL"
LOCAL_ROOT = "./DFL"
api = HfApi(token=HF_TOKEN) # 👈 必须添加这一行
# 🔥 修正：SAVE_PATH 必须与 snapshot_download 的结构完全对应
# 如果仓库里文件夹叫 Results_transformer_Comparison，这里就不要加 FL_Experiments
SAVE_PATH = os.path.join(LOCAL_ROOT, "FL_Experiments/Results_transformer_Comparison_inew")
os.makedirs(SAVE_PATH, exist_ok=True)

print("🔄 正在从 Hugging Face 获取历史文件列表（不下载文件内容）...")

try:
    # 直接请求仓库文件列表
    repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

    # 过滤并提取我们关心目录下的历史实验文件名
    target_dir = "FL_Experiments/Results_transformer_Comparison_inew"
    existing_remote_files = set(
        os.path.basename(f) for f in repo_files
        if f.startswith(target_dir) and f.endswith(".csv")
    )
    print(f"📊 成功获取列表，云端共有 {len(existing_remote_files)} 个历史实验文件。")
except Exception as e:
    print(f"⚠️ 获取云端文件列表失败: {e}")
    existing_remote_files = set()

# 这里不再需要 concat 所有的 CSV 建立 df_history（除非你的代码逻辑强依赖 df_history 的内部数据）
# 如果只是为了查重，后续直接检查文件名即可。
# 加载本地已有数据建立 df_history (用于逻辑判断)
existing_files = glob.glob(os.path.join(SAVE_PATH, "final_Transformer*.csv"))
if existing_files:
    df_history = pd.concat([pd.read_csv(f) for f in existing_files], ignore_index=True)
    print(f"📊 已加载 {len(existing_files)} 个历史实验文件。")
else:
    df_history = pd.DataFrame()
original_stdout = sys.stdout
original_stderr = sys.stderr
import sys # 确保导入了 sys

class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def isatty(self):
        # 委托给原始终端，告知调用者当前是否为交互式终端
        return self.terminal.isatty()

    def close(self):
        if self.log:
            self.log.close()
def deep_clean():
    # 1. 清理 PyTorch 显存缓存
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # 2. 强制触发 Python 垃圾回收
    gc.collect()
    # 3. (可选) 对于极其严重的内存泄漏，可以尝试重置当前进程的上下文
    # 但通常前两步配合 del 就足够了
# 2. 全局配置
NUM_CLIENTS = 20
MALICIOUS_RATIO = 0.3
GLOBAL_ROUNDS = 15
# 1. Data & Topology

MODEL_CHECKPOINT = "distilbert-base-uncased"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
DATA_PATH = os.path.join(LOCAL_ROOT, "data/pubmed")
train_ds, test_ds = get_data(
    dataset_name='pubmed',
    tokenizer=TOKENIZER,
    repo_id=REPO_ID  # 👈 这里改用本地同步后的路径
)
test_loader = DataLoader(test_ds, batch_size=256)
client_datasets = distribute_data(train_ds, NUM_CLIENTS)
#client_datasets = distribute_data_one_class(train_ds, NUM_CLIENTS)
topology_type = 'scale_free'
G = generate_topology(NUM_CLIENTS, topology_type)
neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}
placement_strategy='Topology-Aware'
# 2. 恶意节点
num_mal = int(NUM_CLIENTS * MALICIOUS_RATIO)

defense_budget  = count_minimum_required_defenders(G)#int(NUM_CLIENTS * 0.2)

bf = 0.5
intensity = 0.02




# ==========================================
# 1. Hugging Face
# ==========================================
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "JONESMITH007/DFL"
LOCAL_ROOT = "./DFL"
api = HfApi(token=HF_TOKEN) # 👈 必须添加这一行

sys.stdout = original_stdout
import time

def upload_with_retry(path, repo_path, max_retries=3):
    for i in range(max_retries):
        try:
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=repo_path,
                repo_id=REPO_ID,
                repo_type="dataset"
            )
            print(f"✅ 上传成功: {os.path.basename(path)}")
            return True
        except Exception as upload_err:
            print(f"⚠️ 第 {i+1} 次上传失败 ({os.path.basename(path)}): {upload_err}")
            if i < max_retries - 1:
                time.sleep(5)  # 等待5秒后重试
            else:
                print(f"❌ 最终上传放弃: {os.path.basename(path)}")
                return False

# ==========================================
# 1. Experiment settup
# ==========================================

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
bf = 1.5
intensity = 1
norm_factor = 15
placement_strategy = 'Topology-Aware'

import pytz
from datetime import datetime


tz = pytz.timezone('Asia/Shanghai')

MALICIOUS_RATIOS = [0.25]

TOPOLOGY_TYPES = [ 'random_regular','scale_free']
MALICIOUS_RATIOS = [0.3, 0.1, 0.2]
DEFENSE_RATIOS = [ 0.2]
SEEDS = [1, 2, 3]
MECHANISMS =  ['MAB',  'FLAME','CosL2', 'FedAvg','TrimmedMean', 'Krum']
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

for topo_type in TOPOLOGY_TYPES:
    for ratio in MALICIOUS_RATIOS:
        for def_ratio in DEFENSE_RATIOS:

            num_mal = int(NUM_CLIENTS * ratio)
            current_defense_budget = int(NUM_CLIENTS * def_ratio)

            for seed_idx, current_seed in enumerate(SEEDS):
                for mech in MECHANISMS:


                    csv_filename = f"final_Transformer_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_seed{current_seed}.csv"
                    log_filename = f"Log_Transformer_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_seed{current_seed}.txt"

                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_full_path = os.path.join(SAVE_PATH, log_filename)


                    # if os.path.exists(full_save_path) or csv_filename in existing_remote_files:
                    #     print(f"⏩ 跳过已存在任务: {csv_filename}")
                    #     continue

                    all_results = []


                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger

                    try:
                        print(f"\n{'='*60}")
                        print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print(f"📡 : Topo={topo_type}, Mal={ratio}, Def={def_ratio}, Seed={current_seed}")
                        print(f"{'='*60}")

                        set_seed(current_seed)
                        print(f"▶️ 运行: {mech} | Seed: {current_seed} ({seed_idx+1}/{len(SEEDS)})")


                        G = generate_topology(NUM_CLIENTS, topo_type)
                        neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                        malicious_clients, defense_nodes = allocate_malicious_nodes(
                            G, num_mal, current_defense_budget, topology_type=topo_type, placement= 'Topology-Aware'
                        )

                        theo_intensities = calculate_theoretical_intensity(
                            neighbors, malicious_clients, NUM_CLIENTS, bf, lambda_benign=0.3
                        )

                        start_tick = time.time()
                        start_wall_time = datetime.now(pytz.timezone('Asia/Shanghai')).strftime("%Y-%m-%d %H:%M:%S")

                        ctx = mp.get_context('spawn')


                        with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as executor:

                            future = executor.submit(
                                run_simulation_transformer,
                                current_seed, NUM_CLIENTS, defense_nodes, malicious_clients,
                                G, neighbors, client_datasets, test_ds,
                                #
                                mechanism=mech, bf=bf, intensity=intensity,
                                debug=False, GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                                norm_factor=norm_factor, epochs=1
                            )

                            #
                            _, _, accs, asrs = future.result()
                        shanghai_tz = pytz.timezone('Asia/Shanghai')
                        print(f"🚀 Running: {mech}")
                        end_tick = time.time()
                        duration_sec = round(end_tick - start_tick, 2)
                        start_wall_time = datetime.now(pytz.timezone('Asia/Shanghai')).strftime("%Y-%m-%d %H:%M:%S")

                        # --- 收集结果 ---
                        result_entry_base = {
                            'seed': current_seed, 'mechanism': mech, 'malicious_ratio': ratio,
                            'defense_ratio': def_ratio, 'topology': topo_type,
                            'start_time': start_wall_time, 'duration_sec': duration_sec
                        }

                        for i in range(NUM_CLIENTS):
                            client_row = copy.deepcopy(result_entry_base)
                            client_row.update({
                                'client_id': i, 'final_acc': accs[i], 'final_asr': asrs[i],
                                'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                                'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                            })
                            all_results.append(client_row)

                        #
                        pd.DataFrame(all_results).to_csv(full_save_path, index=False)

                        #
                        sys.stdout = original_stdout
                        print(f"☁️ 正在上传结果至 HF: {csv_filename}")

                        #
                        print(f"☁️ 准备上传结果至 HF...")
                        csv_repo_path = f"FL_Experiments/Results_transformer_Comparison_inew/{csv_filename}"
                        log_repo_path = f"FL_Experiments/Results_transformer_Comparison_inew/{log_filename}"

                        #
                        upload_with_retry(full_save_path, csv_repo_path)
                        upload_with_retry(log_full_path, log_repo_path)

                        sys.stdout = logger

                    except Exception as e:
                        sys.stdout = original_stdout
                    finally:
                        sys.stdout = original_stdout
                        sys.stderr = original_stderr
                        logger.close()

                    deep_clean()


print(f"\n🎉Experiments compeleted: {SAVE_PATH}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔄 正在从 Hugging Face 获取历史文件列表（不下载文件内容）...
📊 成功获取列表，云端共有 126 个历史实验文件。


Fetching ... files: 0it [00:00, ?it/s]

Map (num_proc=11):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=11):   0%|          | 0/1000 [00:00<?, ? examples/s]


⏰ : 2026-03-15 06:38:23
📡 : Topo=random_regular, Mal=0.3, Def=0.2, Seed=1
▶️ 运行: FedAvg | Seed: 1 (1/3)
  [Critical Warning] Topology constraints are too strict! Can only safely allocate 5/6 malicious clients.
🚀 Running: FedAvg
☁️ 正在上传结果至 HF: final_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed1.csv
☁️ 准备上传结果至 HF...
✅ 上传成功: final_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed1.csv
✅ 上传成功: Log_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed1.txt

⏰ : 2026-03-15 06:43:33
📡 : Topo=random_regular, Mal=0.3, Def=0.2, Seed=2
▶️ 运行: FedAvg | Seed: 2 (2/3)
🚀 Running: FedAvg
☁️ 正在上传结果至 HF: final_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed2.csv
☁️ 准备上传结果至 HF...
✅ 上传成功: final_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed2.csv
✅ 上传成功: Log_Transformer_random_regular_MR0.3_DR0.2_FedAvg_seed2.txt

⏰ : 2026-03-15 06:48:47
📡 : Topo=random_regular, Mal=0.3, Def=0.2, Seed=3
▶️ 运行: FedAvg | Seed: 3 (3/3)
🚀 Running: FedAvg
☁️ 正在上传结果至 HF: final_Transformer_random_regular_MR0.3_DR0.2_F